# 03 · Entrenamiento del Modelo de Clustering
## Segmentación Inteligente de Pacientes — Comfama

| | |
|---|---|
| **Propósito** | Entrenar KMeans sobre las variables de estilo de vida, seleccionar *k* y registrar el modelo en MLflow / Unity Catalog. |
| **Entrada** | Tabla Delta `<catalog>.<schema>.fenotipo_features` (notebook 02). |
| **Salida** | Experimento MLflow + modelo registrado en UC (`<catalog>.<schema>.fenotipo_clustering`). |
| **Siguiente paso** | `04_perfilamiento_inferencia`. |

---
### 🧠 Enfoque metodológico (trazabilidad de decisiones)
- **Algoritmo:** `KMeans` (interpretable, eficiente y estándar para segmentación de clientes/pacientes).
- **Pipeline versionado:** `ColumnTransformer` (StandardScaler en numéricas + OneHotEncoder en categóricas) → `KMeans`. La transformación **viaja con el modelo**, de modo que la inferencia es idéntica al entrenamiento.
- **Selección de *k*:** priorizamos el **coeficiente de silueta** (↑ mejor), acompañado de **Davies-Bouldin** (↓ mejor), **Calinski-Harabasz** (↑ mejor) e **inercia** (método del codo).
- **Rango explorado:** *k* = 2..8, alineado con el objetivo de negocio de identificar *subgrupos emergentes*.

### Paso 1 · Parámetros

In [ ]:
dbutils.widgets.text("catalog", "main", "Catálogo")
dbutils.widgets.text("schema", "fenotipo", "Esquema")
dbutils.widgets.text("k_min", "2", "k mínimo")
dbutils.widgets.text("k_max", "8", "k máximo")
dbutils.widgets.text("k_final", "0", "k final (0 = mejor silueta)")
dbutils.widgets.text("model_name", "main.fenotipo.fenotipo_clustering", "Nombre del modelo (UC)")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
K_MIN = int(dbutils.widgets.get("k_min"))
K_MAX = int(dbutils.widgets.get("k_max"))
K_FINAL = int(dbutils.widgets.get("k_final"))
MODEL_NAME = dbutils.widgets.get("model_name")
FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.fenotipo_features"

### Paso 2 · Carga de features y contrato de variables
Reutilizamos el **mismo catálogo de variables** definido en el notebook 02 (consistencia end-to-end).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

NUMERIC_FEATURES = [
    "habitos_alim_cantidad",
    "hab_dejar_endulzar", "hab_dejar_reposteria", "hab_incorporar_vegetales",
    "hab_ajustar_carbohidratos", "hab_aumentar_proteina", "hab_otros",
    "med_hipoglicemiantes", "med_hipolipemiantes", "med_ninguno",
]
CATEGORICAL_FEATURES = [
    "actividad_tipo", "actividad_duracion", "actividad_frecuencia",
    "estres_altas_cargas", "estres_tecnica_manejo",
]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

pdf = spark.table(FEATURES_TABLE).toPandas()
X = pdf[ALL_FEATURES].copy()
for c in NUMERIC_FEATURES:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)
print(f"Muestras: {len(X):,} | Features: {len(ALL_FEATURES)}")

### Paso 3 · Transformador de features
Función reutilizable que escala numéricas y codifica categóricas. Incluye compatibilidad con versiones de scikit-learn (`sparse_output` vs `sparse`).

In [ ]:
def build_preprocessor() -> ColumnTransformer:
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # sklearn < 1.2
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", ohe, CATEGORICAL_FEATURES),
    ])

### Paso 4 · Selección de *k*
Entrenamos KMeans para cada *k* del rango y comparamos las métricas. **Interpretación:** silueta y Calinski-Harabasz mayores = mejor separación; Davies-Bouldin menor = mejor.

In [ ]:
pre = build_preprocessor()
X_trans = pre.fit_transform(X)

resultados = []
for k in range(K_MIN, K_MAX + 1):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_trans)
    resultados.append({
        "k": k,
        "inercia": km.inertia_,
        "silueta": silhouette_score(X_trans, labels),
        "davies_bouldin": davies_bouldin_score(X_trans, labels),
        "calinski_harabasz": calinski_harabasz_score(X_trans, labels),
    })

res_df = pd.DataFrame(resultados)
display(res_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(res_df["k"], res_df["inercia"], "o-"); axes[0].set_title("Codo (inercia)"); axes[0].set_xlabel("k")
axes[1].plot(res_df["k"], res_df["silueta"], "o-", color="#FF277E"); axes[1].set_title("Silueta (↑ mejor)"); axes[1].set_xlabel("k")
axes[2].plot(res_df["k"], res_df["davies_bouldin"], "o-", color="#7B4F9E"); axes[2].set_title("Davies-Bouldin (↓ mejor)"); axes[2].set_xlabel("k")
plt.tight_layout(); plt.show()

> **Nota práctica:** en la validación con datos reales la silueta favorece `k=2`. Si el objetivo es exponer **subgrupos accionables** para el equipo clínico, fija `k_final` (p. ej. 4) en el widget y documenta la decisión.

In [ ]:
if K_FINAL > 0:
    k_best = K_FINAL
    print(f"k seleccionado por decisión de negocio (widget): {k_best}")
else:
    k_best = int(res_df.loc[res_df["silueta"].idxmax(), "k"])
    print(f"k seleccionado por mejor silueta: {k_best}")

### Paso 5 · Entrenamiento final y registro en MLflow / Unity Catalog
Registramos parámetros, métricas, **firma** y ejemplo de entrada. El modelo queda versionado en UC para gobierno y despliegue.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Shared/fenotipo_clustering")

with mlflow.start_run(run_name=f"kmeans_k{k_best}") as run:
    pipe = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("kmeans", KMeans(n_clusters=k_best, random_state=42, n_init=10)),
    ])
    cluster_labels = pipe.fit_predict(X)
    Xt = pipe.named_steps["preprocessor"].transform(X)

    mlflow.log_params({
        "algoritmo": "KMeans", "k": k_best,
        "n_features_num": len(NUMERIC_FEATURES),
        "n_features_cat": len(CATEGORICAL_FEATURES),
        "n_muestras": len(X),
    })
    mlflow.log_metrics({
        "silueta": float(silhouette_score(Xt, cluster_labels)),
        "davies_bouldin": float(davies_bouldin_score(Xt, cluster_labels)),
        "calinski_harabasz": float(calinski_harabasz_score(Xt, cluster_labels)),
        "inercia": float(pipe.named_steps["kmeans"].inertia_),
    })

    signature = infer_signature(X, cluster_labels)
    mlflow.sklearn.log_model(
        sk_model=pipe, artifact_path="model", signature=signature,
        input_example=X.head(3), registered_model_name=MODEL_NAME,
    )
    run_id = run.info.run_id
    print(f"✅ Run: {run_id}\n✅ Modelo registrado: {MODEL_NAME}")

### Paso 6 · Distribución de clusters del modelo final

In [ ]:
pdf["cluster"] = cluster_labels
display(pdf["cluster"].value_counts().sort_index().rename_axis("cluster").reset_index(name="n_pacientes"))
print("Sugerencia: usa este run_id / modelo en el notebook 04:", run_id)

---
**Resumen del notebook 03:** modelo KMeans entrenado, evaluado y registrado en UC.
➡️ Continúa en **`04_perfilamiento_inferencia`** para interpretar y aplicar el modelo.